In [ ]:
import arviz as az
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import Markdown

from emu_renewal.inputs import ANALYSIS_TYPES, get_latest_analyses
from emu_renewal.outputs import get_output_dir, melt_df_except_first_level, get_multianalysis_dispvals_from_idatas, get_multianalysis_procvals_from_idatas, get_multianalysis_likelihoods
from emu_renewal.outputs import get_multianalysis_ind_spaghetti, get_multianalysis_procvals
from emu_renewal.plotting import plot_process_comparison, plot_updates_comparison, plot_mob_update_comparison

set_matplotlib_formats("svg")

In [ ]:
country = "Sweden"
all_analysis_times = get_latest_analyses(country, ANALYSIS_TYPES)

In [ ]:
#| fig-cap: "Variable process residual function by inclusion of candidate mobility modifiers."
spagh_df = get_multianalysis_ind_spaghetti(country, "process", all_analysis_times)
proc_fig = plot_process_comparison(spagh_df, all_analysis_times.keys(), ["k", "cornflowerblue", "mediumblue", "red", "darkred"], 0.1);
proc_fig.legend(bbox_to_anchor=[0.85, 0.75], loc="right")

In [ ]:
#| fig-cap: "Variable process update values by inclusion of candidate mobility modifiers."
idatas = {k: az.from_netcdf(get_output_dir(country, k, v) / "idata.nc") for k, v in all_analysis_times.items()}
update_fig = plot_mob_update_comparison(idatas, 0.3, fig_height=14)

In [ ]:
multianalysis_proc_df = get_multianalysis_procvals_from_idatas(idatas)
multianalysis_disp_df = get_multianalysis_dispvals_from_idatas(idatas)

In [ ]:
# | fig-cap: "Dispersion parameter posteriors by inclusion of candidate mobility modifiers. Lower values for the dispersion parameter allow for smaller updates to the variable process over time."
disp_fig, ax = plt.subplots(figsize=(6, 6))
disp_comparison_df = melt_df_except_first_level(multianalysis_disp_df)
sns.kdeplot(disp_comparison_df, fill=True, ax=ax)
ax.set_xlim([0.0, 0.25]);
ax.set_title(country);
disp_fig.savefig(f"{country.lower()}_disp_fig.svg")

In [ ]:
#| fig-cap: "Comparison of likelihoods by inclusion of candidate mobility modifiers."
like_fig, ax = plt.subplots(figsize=(6, 6))
likelihoods = melt_df_except_first_level(get_multianalysis_likelihoods(country, all_analysis_times))
sns.kdeplot(likelihoods, fill=True, ax=ax)
ax.set_title(country);
like_fig.savefig(f"{country.lower()}_like_fig.svg")

In [ ]:
#| tbl-cap: "Median likelihood values by inclusion of candidate mobility modifiers."
medians = likelihoods.median()
medians.index.name = "Analysis type"
medians.name = "Median values"
Markdown(medians.to_markdown())